# Set 10 – Support Vector Classification (SVC)

Wir bauen eine vollständige SVM-Klassifikation auf kontrollierten 2D-Daten. So werden Trennfläche, Margin und Support-Vektoren sichtbar.

## Lernziele

- Margin, Soft Margin und Support-Vektoren verstehen
- `SVC` in einer `Pipeline` verwenden
- `fit`, `predict` und `decision_function` unterscheiden
- linearen, polynomialen und RBF-Kernel vergleichen
- `C` und `gamma` per Cross-Validation optimieren

SVMs arbeiten mit Abständen. Deshalb gehört der `StandardScaler` in die Pipeline: Er verhindert, dass Maßeinheiten die Geometrie verzerren, und schützt in der Cross-Validation vor Data Leakage.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
RANDOM_STATE=42
plt.style.use("seaborn-v0_8-whitegrid")
from sklearn.datasets import make_moons
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay


## 1. Daten erzeugen

`make_moons` erzeugt zwei gebogene Klassen. `noise` regelt die Überlappung. Der stratifizierte Split erhält die Klassenanteile.


In [ ]:
X,y=make_moons(n_samples=500,noise=.22,random_state=RANDOM_STATE)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=RANDOM_STATE)
plt.figure(figsize=(7,5)); plt.scatter(X_train[:,0],X_train[:,1],c=y_train,cmap="coolwarm",s=28,alpha=.8)
plt.xlabel("Merkmal 1"); plt.ylabel("Merkmal 2"); plt.title("Trainingsdaten"); plt.show()
print("Training:",X_train.shape,"Test:",X_test.shape)


## 2. `SVC`: wichtige Parameter und Methoden

| Name | Bedeutung |
|---|---|
| `kernel` | `linear`, `poly`, `rbf` oder `sigmoid` |
| `C` | Fehlerstrafe: klein = stärkere Regularisierung; groß = Trainingsfehler teurer |
| `gamma` | Reichweite bei RBF/Poly: klein = glatt; groß = lokal und komplex |
| `degree` | Polynomgrad, nur bei `poly` |
| `class_weight` | z. B. `balanced` für ungleiche Klassenkosten |
| `probability` | aktiviert `predict_proba`, erhöht den Trainingsaufwand |
| `fit(X,y)` | lernt Scaler und Modell |
| `predict(X)` | gibt Klassenlabels aus |
| `decision_function(X)` | vorzeichenbehafteter Abstand zur Grenze |

`gamma="scale"` berechnet einen datenabhängigen Startwert. Beim linearen Kernel hat `gamma` keine Wirkung.


In [ ]:
model=Pipeline([("scaler",StandardScaler()),("svc",SVC(kernel="rbf",C=1,gamma="scale"))])
model.fit(X_train,y_train)
pred=model.predict(X_test); scores=model.decision_function(X_test)
print("Labels:",pred[:8]); print("Decision Scores:",np.round(scores[:8],3))
print("Accuracy:",round(accuracy_score(y_test,pred),3),"F1:",round(f1_score(y_test,pred),3))


## 3. Margin und Support-Vektoren

Decision Score 0 ist die Trennfläche, −1/+1 markieren die Margins. Punkte an oder innerhalb der Margin tragen die Lösung als **Support-Vektoren**. `support_` sind Trainingsindizes, `support_vectors_` die skalierten Werte und `n_support_` die Anzahl je Klasse.


In [ ]:
def plot_boundary(model,X,y,title,ax=None):
 if ax is None: _,ax=plt.subplots(figsize=(7,5))
 xx,yy=np.meshgrid(np.linspace(X[:,0].min()-.5,X[:,0].max()+.5,240),np.linspace(X[:,1].min()-.5,X[:,1].max()+.5,240))
 z=model.decision_function(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
 ax.contourf(xx,yy,z>0,alpha=.18,cmap="coolwarm")
 ax.contour(xx,yy,z,levels=[-1,0,1],colors=["grey","black","grey"],linestyles=["--","-","--"])
 ax.scatter(X[:,0],X[:,1],c=y,cmap="coolwarm",s=24,alpha=.72)
 support=model.named_steps["scaler"].inverse_transform(model.named_steps["svc"].support_vectors_)
 ax.scatter(support[:,0],support[:,1],s=85,facecolors="none",edgecolors="black",label="Support-Vektoren")
 ax.set(title=title,xlabel="Merkmal 1",ylabel="Merkmal 2"); ax.legend()
plot_boundary(model,X_train,y_train,"RBF-SVC: Grenze und Margin"); plt.show()
print("Support-Vektoren je Klasse:",model.named_steps["svc"].n_support_)


## 4. Kernels vergleichen

- `linear`: Hyperebene; schnell und vergleichsweise interpretierbar
- `poly`: gekrümmte Beziehungen, gesteuert durch `degree`
- `rbf`: flexible lokale Ähnlichkeit

Mehr Flexibilität ist nicht automatisch besser. Entscheidend ist Leistung auf unbekannten Daten.


In [ ]:
estimators={"Linear":SVC(kernel="linear",C=1),"Poly":SVC(kernel="poly",degree=3,C=1),"RBF":SVC(kernel="rbf",C=1)}
fig,axes=plt.subplots(1,3,figsize=(18,4.8)); rows=[]
for ax,(name,est) in zip(axes,estimators.items()):
 pipe=Pipeline([("scaler",StandardScaler()),("svc",est)]).fit(X_train,y_train); p=pipe.predict(X_test)
 rows.append({"Kernel":name,"Accuracy":accuracy_score(y_test,p),"F1":f1_score(y_test,p),"Support":pipe.named_steps["svc"].support_.size})
 plot_boundary(pipe,X_train,y_train,name,ax)
plt.tight_layout(); plt.show(); display(pd.DataFrame(rows).round(3))


## 5. `C` und `gamma`

Kleines `C` toleriert Fehler; großes `C` passt Trainingspunkte stärker an. Kleines `gamma` erzeugt eine glatte, weitreichende Grenze; großes `gamma` eine lokale, eventuell zerklüftete Grenze. Sehr große Werte beider Parameter können overfitten.


In [ ]:
settings=[(.1,.1),(.1,10),(100,.1),(100,10)]; fig,axes=plt.subplots(2,2,figsize=(12,9))
for ax,(C_value,gamma_value) in zip(axes.ravel(),settings):
 pipe=Pipeline([("scaler",StandardScaler()),("svc",SVC(kernel="rbf",C=C_value,gamma=gamma_value))]).fit(X_train,y_train)
 plot_boundary(pipe,X_train,y_train,f"C={C_value}, gamma={gamma_value}",ax)
plt.tight_layout(); plt.show()


## 6. Hyperparameter-Suche

Die Liste von Suchräumen testet nur zum Kernel passende Parameter. `svc__` adressiert den Pipeline-Schritt. Die Suche läuft nur auf Trainingsdaten; Testdaten bleiben bis zum Schluss unangetastet.


In [ ]:
pipe=Pipeline([("scaler",StandardScaler()),("svc",SVC())])
grid=[{"svc__kernel":["linear"],"svc__C":[.1,1,10]},
{"svc__kernel":["poly"],"svc__C":[.1,1,10],"svc__gamma":["scale",.1],"svc__degree":[2,3]},
{"svc__kernel":["rbf"],"svc__C":[.1,1,10],"svc__gamma":[.01,.1,1]}]
search=GridSearchCV(pipe,grid,scoring="f1",cv=5,n_jobs=-1,return_train_score=True).fit(X_train,y_train)
print("Beste Parameter:",search.best_params_); print("Bester CV-F1:",round(search.best_score_,3))
res=pd.DataFrame(search.cv_results_); display(res[["params","mean_train_score","mean_test_score","std_test_score"]].sort_values("mean_test_score",ascending=False).head(10).round(3))


In [ ]:
best=search.best_estimator_; p=best.predict(X_test)
print(classification_report(y_test,p,digits=3)); ConfusionMatrixDisplay.from_predictions(y_test,p,cmap="Blues"); plt.show()
plot_boundary(best,X_train,y_train,"Bestes Modell"); plt.show()


## Fazit

SVMs maximieren die Margin; Support-Vektoren tragen die Grenze. Skalierung ist zentral. `C` kontrolliert Fehlerstrafen, `gamma` die Reichweite nichtlinearer Kernels. Auswahl per Cross-Validation, Schlussbewertung auf Testdaten.
